# stage4 第三輪-B:補 text 落地(A)+ 調重訓再試(B)

**本機實證已定論:**
- 退步元兇 = **降採樣**(誤砍 50% ab_eval semantic GT 正樣本),非補 text。
- **補 text 幾乎無害**(semantic −0.03)且提升設施隱喻(報稅 0.40→0.97、車位 0.17→0.43)。
- 『只補 text 不重訓』本機 onnx 編向量已驗:ab_eval all R@30=0.254(守 0.26)+ semantic 0.46 不退。

**兩產物(一次跑完比較):**
- **A 補 text 落地**:現役等價模型(原始資料重訓+蒸餾)+ 富化 text → 正式 torch 向量 = 穩贏。
- **B 調重訓**:原始資料 + 設施 pair(**不降採樣**)+ 富化 text 重訓 → 試設施拿 1.0 且 semantic 不崩。


## 0. 環境


In [ ]:
!nvidia-smi -L
%cd /content
!rm -rf nchu-edge-rental-ai
!git clone -q https://github.com/eric20041027/nchu-edge-rental-ai.git
%cd nchu-edge-rental-ai
!pip -q install torch transformers onnxruntime onnx tokenizers numpy


## 1. 補特徵線索進 text(A、B 共用;台電排除)


In [ ]:
!python -m pipeline.data_prep.enrich_text_features --write


## 2. 重訓現役等價 rbt6 teacher(原始資料,無設施 pair / 無降採樣)= A 模型 + B teacher


In [ ]:
!python -m pipeline.model_training.train_bi_encoder \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 64 --bf16 --tf32 \
    --output-dir saved_models/rbt6_base
!python -m pipeline.model_training.distill_bi_encoder \
    --teacher-dir saved_models/rbt6_base --output-dir saved_models/rbt3_base \
    --epochs 3 --batch-size 32 --lr 3e-5 --alpha 0.5 --max-length 64 --bf16 --tf32


## 3-A. 【產物 A】現役等價模型 + 富化 text → 正式向量 + 雙 gate


In [ ]:
!python -m pipeline.model_training.export_bi_encoder --saved-dir saved_models/rbt3_base
!python -m pipeline.data_prep.build_property_embeddings --saved-dir saved_models/rbt3_base
!python -m pipeline.data_prep.build_property_embeddings --check
print('=== A Gate1: ab_eval(守 0.26 不退步)===')
!python pipeline/evaluation/eval_vector_vs_rulebased.py 2>&1 | grep -E 'semantic \(78\)|all \(278\)|Recall@30|GO/NO'
print('=== A Gate2: #99 設施標靶 ===')
!python pipeline/evaluation/eval_hidden_meaning.py 2>&1 | tail -6
!tar czf /content/A_enrich_only.tgz frontend/models/bi_encoder_dir/bi_encoder_quant.onnx frontend/assets/property_embeddings.json frontend/assets/property_data.json
from google.colab import files; files.download('/content/A_enrich_only.tgz')


## 3-B. 【產物 B】原始資料 + 設施 pair(不降採樣)+ 富化 text 重訓


In [ ]:
import json
!python -m pipeline.data_prep.gen_facility_intent_data --per 8
orig = json.load(open('data/processed/recommendation_train.json'))
fac  = json.load(open('data/processed/facility_intent_train.json'))
merged = orig + fac
json.dump(merged, open('data/processed/train_merged_b.json','w'), ensure_ascii=False)
print(f'B 訓練集: {len(orig)} 原始(不降採樣) + {len(fac)} 設施 = {len(merged)}')


In [ ]:
!TRAIN_DATA_PATH=data/processed/train_merged_b.json \
    python -m pipeline.model_training.train_bi_encoder \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 64 --bf16 --tf32 \
    --output-dir saved_models/rbt6_b
!TRAIN_DATA_PATH=data/processed/train_merged_b.json \
    python -m pipeline.model_training.distill_bi_encoder \
    --teacher-dir saved_models/rbt6_b --output-dir saved_models/rbt3_b \
    --epochs 3 --batch-size 32 --lr 3e-5 --alpha 0.5 --max-length 64 --bf16 --tf32


In [ ]:
!python -m pipeline.model_training.export_bi_encoder --saved-dir saved_models/rbt3_b
!python -m pipeline.data_prep.build_property_embeddings --saved-dir saved_models/rbt3_b
print('=== B Gate1: ab_eval(不降採樣應不崩 semantic)===')
!python pipeline/evaluation/eval_vector_vs_rulebased.py 2>&1 | grep -E 'semantic \(78\)|all \(278\)|Recall@30|GO/NO'
print('=== B Gate2: #99 設施標靶(試拿 1.0)===')
!python pipeline/evaluation/eval_hidden_meaning.py 2>&1 | tail -6
!tar czf /content/B_facility_retrain.tgz frontend/models/bi_encoder_dir/bi_encoder_quant.onnx frontend/assets/property_embeddings.json frontend/assets/property_data.json
from google.colab import files; files.download('/content/B_facility_retrain.tgz')


## 落地決策(本機複核 A、B 後)
- **A**:ab_eval all R@30 ≥ 0.26 + #99 設施提升 → 穩贏,優先落地。
- **B**:semantic 不崩(不降採樣修好)且設施更接近 1.0 → 落地 B;仍崩 → 落地 A。
- 不盲信 Colab,本機 eval_vector_vs_rulebased + eval_hidden_meaning + preview 親驗才落地。
